# 02 — Continued pre-training (CPT) on CPU

**SFT** (notebook 01) teaches a model a behavior from request → answer pairs. **Continued pre-training** teaches
it *knowledge* from plain text: the same next-token objective the model was pre-trained with, on documents from
your domain, with no chat template and no answers.

The domain here is a fictional restaurant, *Trattoria Lumen* (its chef, opening hours, signature dish, ...), so
the base model cannot know any of it. The job:

1. asks the base model 8 questions about the restaurant, phrased differently from the training text
   (fill-in-the-blank), and measures its perplexity on 8 held-out sentences
2. trains a LoRA adapter on 200 short generated documents (60 steps, LoRA r=16 on attention + MLP)
3. asks the same questions and measures perplexity again

A high score after training means the model learned the facts, not the sentences: the questions never appear in
the training text. Expected: ~5 minutes, mostly job startup.

In [ ]:
%pip install -q "sagemaker==3.21.0"

## Setup

`Lab` (in `sm_helpers.py`) holds the steps notebook 01 spells out: the SageMaker session, execution role and S3
bucket, MLflow, and a CPU training job. Two settings:

* `CAPACITY`: how the job gets its instance: `"fallback"` (the first of several CPU types with capacity; the
  default), `"on-demand"` (exactly `ml.c7i.4xlarge`) or `"spot"` (`ml.m5.4xlarge`, cheaper, may wait for capacity). Notebook 01
  compares the three.
* `MLFLOW_APP_ARN`: where the job logs its metrics (see *Track in MLflow* below).

In [ ]:
import sys

sys.path.insert(0, "scripts")
from sm_helpers import Lab

# MLflow: "" = reuse the account's MLflow App (or create one: several minutes, first time only);
# "arn:aws:sagemaker:..." = use that app; None = off. Jobs run the same either way.
MLFLOW_APP_ARN = ""
lab = Lab(mlflow_app_arn=MLFLOW_APP_ARN)
CAPACITY = "fallback"   # "on-demand" | "spot" | "fallback" (compared side by side in 01-sft.ipynb)

In [ ]:
import booking_data

print(booking_data.cpt_documents(n_docs=1)[0], "\n")
for key in ("chef", "closed_day"):
    print(f"{booking_data.CLOZE[key]} ...   (expected: {booking_data.FACTS[key]})")

## Train

One training job runs `scripts/train_cpt.py`: it generates the restaurant documents, scores the base model, trains
the adapter for 60 steps with the plain next-token loss, and scores again. The next cell waits and prints status
changes; most of the ~5 minutes is the instance starting and the model downloading. If the kernel restarts,
re-run the setup cell and the wait cell: it finds the latest job.

In [ ]:
job = lab.submit("cpt", "train_cpt.py", {"steps": 60, "model_id": "Qwen/Qwen3-0.6B"}, capacity=CAPACITY)

In [ ]:
job = globals().get("job") or lab.latest("cpt")   # after a kernel restart
desc = lab.wait(job)

## Results

In [ ]:
m = lab.metrics(job)
print(f"{'':10} {'questions answered':>19} {'perplexity (held-out)':>22}")
for which in ("base", "finetuned"):
    print(f"{which:10} {m[which]['cloze_accuracy']:19.0%} {m[which]['heldout_perplexity']:22.1f}")
print()
for ex in m["examples"]:
    print(f"{ex['question']}\n   base:      {ex['base']!r}\n   finetuned: {ex['finetuned']!r}\n")

### How to read the results

| Row | Model |
|---|---|
| `base` | Qwen3-0.6B as downloaded, before training |
| `finetuned` | the same model + the LoRA adapter trained on the restaurant documents |

| Column | What it measures | Expected |
|---|---|---|
| questions answered | 8 fill-in-the-blank sentences about the restaurant (e.g. *"The head chef at Trattoria Lumen is ..."*), worded differently from every training document. A question counts as answered if the model's continuation contains the right fact | base ~0% (it cannot know a fictional restaurant), fine-tuned close to 100% |
| perplexity (held-out) | how surprised the model is by 8 true sentences about the restaurant that were never in training: e<sup>average loss per token</sup>. 1 = perfectly predicted; the base model's value is that of plausible but unknown text | drops sharply (e.g. ~150 → under 10) |

The examples below the table show the base model inventing a plausible answer and the fine-tuned model giving
the fact from the documents.

Both measures use phrasings the model never saw, so a high score means it learned the **facts**, not the
sentences. If only training sentences were tested, a model that memorized word sequences would also score well.

## Track in MLflow

Each job also logs its parameters, the training loss per step, and the base and fine-tuned scores to MLflow,
one run per job in the experiment `cpu-fine-tuning-techniques`. Select runs and **Compare** to put the techniques
side by side. The link below is valid for a few minutes; re-run the cell for a new one.

In [ ]:
# Compare runs in MLflow: experiment "cpu-fine-tuning-techniques", one run per job (named after the job)
print(lab.mlflow_url() or "MLflow is off")

## Takeaway

CPT changes what the model **knows**, not how it behaves: there is no instruction and no answer format in the
training data, only text. In practice it is the first step when a model must speak a domain's language or know
its facts (internal documents, product catalogs, a new language); SFT and preference tuning then shape how it
answers. Real CPT runs use far more text (millions to billions of tokens) and usually full fine-tuning or a high
LoRA rank; this lab shows the mechanics on 200 short documents.